# Skenario 3 — Hyperparameter Tuning & Ensemble
**Tujuan:** Hyperparameter tuning (epoch, batch size, learning rate, dropout) lalu ensemble majority voting

**Task:** Klasifikasi Emosi (5 kelas) dan Sentimen (3 kelas)

---
**Strategi hemat RAM:** Setiap kombinasi hyperparameter × model dilatih satu per satu, hasilnya langsung di-save ke `results_s3.json`.

**🏆 Best model:** Model + tokenizer terbaik per task (across semua HP config) di-save ke `/kaggle/working/best_model_s3_{task}/`

## 0. Install & Import

In [ ]:
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip("transformers[torch]", "datasets", "accelerate", "scikit-learn",
    "imbalanced-learn", "seaborn", "matplotlib",
    "pandas", "numpy", "torch")

In [ ]:
import os, re, json, warnings, random, time, gc
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset as HFDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, f1_score,
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## 1. Load & Preprocessing Dataset

In [ ]:
DATA_PATH = "/kaggle/input/datasets/miskiyah/fp-prdect-id/PRDECT-ID Dataset_clean.csv"

df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

def clean_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"http\S+|www\.\S+", " ", text) 
    text = re.sub(r"[^\w\s]", " ", text)          
    text = re.sub(r"\d+", " ", text)                
    text = re.sub(r"\s+", " ", text).strip()          
    return text

df["text_clean"] = df["Customer Review"].apply(clean_text)

le_sent = LabelEncoder()
le_emot = LabelEncoder()
df["label_sentiment"] = le_sent.fit_transform(df["Sentiment"])
df["label_emotion"]   = le_emot.fit_transform(df["Emotion"])

print("Shape :", df.shape)
print("Label Sentimen :", dict(zip(le_sent.classes_, le_sent.transform(le_sent.classes_))))
print("Label Emosi    :", dict(zip(le_emot.classes_, le_emot.transform(le_emot.classes_))))

TASKS = {
    "Emotion":   ("text_clean", "label_emotion",   le_emot.classes_),
    "Sentiment": ("text_clean", "label_sentiment", le_sent.classes_),
}

## 2. Helper Functions

In [ ]:
def make_splits(texts, labels, test_size=0.2, val_size=0.1):
    X_train, X_test, y_train, y_test = train_test_split(
        texts, labels, test_size=test_size, random_state=SEED, stratify=labels)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=val_size/(1-test_size), random_state=SEED, stratify=y_train)
    return X_train, X_val, X_test, y_train, y_val, y_test


def build_hf_dataset(texts, labels, tokenizer, max_len=128):
    enc = tokenizer(list(texts), truncation=True, padding="max_length",
                    max_length=max_len, return_tensors="pt")
    return HFDataset.from_dict({
        "input_ids":      enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels":         list(labels),
    })


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }


def train_and_evaluate(model_name, train_ds, val_ds, test_texts, test_labels,
                       tokenizer, num_labels, label_names,
                       output_dir="./tmp_model",
                       num_epochs=3, batch_size=16, lr=2e-5,
                       weight_decay=0.01, dropout=None,
                       warmup_ratio=0.1, max_len=128):
    import time
    current_batch_size = batch_size
    while True:
        try:
            # Retry loading model dengan network resilience
            for attempt in range(3):
                try:
                    model = AutoModelForSequenceClassification.from_pretrained(
                        model_name, num_labels=num_labels, ignore_mismatched_sizes=True)
                    break
                except Exception as e:
                    if attempt == 2:
                        raise e
                    print(f"  ⚠️ Gagal download model {model_name}, mencoba lagi ({attempt+2}/3) dalam 5 detik... (Error: {e})")
                    time.sleep(5)

            if dropout is not None and hasattr(model.config, "hidden_dropout_prob"):
                model.config.hidden_dropout_prob = dropout
                model.config.attention_probs_dropout_prob = dropout

            args = TrainingArguments(
                output_dir=output_dir,
                num_train_epochs=num_epochs,
                per_device_train_batch_size=current_batch_size,
                per_device_eval_batch_size=current_batch_size,
                learning_rate=lr,
                weight_decay=weight_decay,
                warmup_ratio=warmup_ratio,
                eval_strategy="epoch",
                save_strategy="epoch",
                save_total_limit=1,
                save_only_model=True,
                load_best_model_at_end=True,
                metric_for_best_model="f1_macro",
                logging_steps=50,
                fp16=(DEVICE == "cuda"),
                report_to="none",
                seed=SEED,
            )

            trainer = Trainer(
                model=model,
                args=args,
                train_dataset=train_ds,
                eval_dataset=val_ds,
                compute_metrics=compute_metrics,
                callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
            )

            t0 = time.time()
            trainer.train()
            elapsed = time.time() - t0
            break
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"  ⚠️ [OOM] CUDA out of memory dengan batch_size={current_batch_size}.")
                if 'trainer' in locals():
                    del trainer
                if 'model' in locals():
                    del model
                free_memory()
                current_batch_size = current_batch_size // 2
                if current_batch_size < 4:
                    print("  ❌ [OOM] Batch size sudah minimum (< 4). Gagal melatih model ini.")
                    raise e
                print(f"  🔄 Mencoba ulang dengan batch_size={current_batch_size}...")
            else:
                raise e

    test_ds = build_hf_dataset(test_texts, test_labels, tokenizer, max_len)
    preds_raw = trainer.predict(test_ds)
    preds_logits = preds_raw.predictions
    if isinstance(preds_logits, tuple):
        preds_logits = preds_logits[0]
    preds = np.argmax(preds_logits, axis=-1)

    metrics = {
        "accuracy":    round(accuracy_score(test_labels, preds), 4),
        "f1_macro":    round(f1_score(test_labels, preds, average="macro"), 4),
        "f1_weighted": round(f1_score(test_labels, preds, average="weighted"), 4),
        "train_time_s": round(elapsed, 1),
    }
    report = classification_report(test_labels, preds, target_names=label_names)
    return trainer, metrics, report, preds


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def save_best_model(trainer, tokenizer, task_name, model_alias, model_name,
                    hp_label, hyperparams, metrics, label_names, skenario="3"):
    """Simpan model + tokenizer + metadata ke folder best_model_s{skenario}_{task}."""
    best_dir = f"/kaggle/working/best_model_s{skenario}_{task_name}"
    os.makedirs(best_dir, exist_ok=True)
    trainer.save_model(best_dir)
    tokenizer.save_pretrained(best_dir)
    meta = {
        "skenario": skenario,
        "task": task_name,
        "model_alias": model_alias,
        "model_name": model_name,
        "hp_label": hp_label,
        "hyperparams": hyperparams,
        "label_names": list(label_names),
        "metrics": metrics,
    }
    with open(f"{best_dir}/best_model_info.json", "w") as f:
        json.dump(meta, f, indent=2)
    print(f"  🏆 [BEST MODEL] {model_alias} @ {hp_label} (f1_macro={metrics['f1_macro']:.4f}) → {best_dir}")

---
# SKENARIO 3 — Hyperparameter Tuning

Setiap konfigurasi hyperparameter × model dilatih satu per satu → **best model per task disimpan** → memory dibebaskan.

In [ ]:
# Model yang diuji (ubah sesuai kebutuhan)
S3_MODELS = {
    "IndoBERT":       "indobenchmark/indobert-base-p1",
    "cahya-DistilBERT": "cahya/distilbert-base-indonesian",
    "cahyaBERT":      "cahya/bert-base-indonesian-522M",
}

# Grid hyperparameter — tiap kombinasi akan dilatih satu per satu
# (label, epochs, batch, lr, dropout)
HYPERPARAM_GRID = [
    ("base",   3, 16, 2e-5, None),   # Baseline
    ("ep5",    5, 16, 2e-5, None),   # Lebih banyak epoch
    ("bs32",   3, 32, 2e-5, None),   # Batch size besar
    ("lr3e5",  3, 16, 3e-5, None),   # LR lebih tinggi
    ("lr1e5",  3, 16, 1e-5, None),   # LR lebih rendah
    ("drop02", 3, 16, 2e-5, 0.2),    # Dropout 0.2
    ("drop03", 3, 16, 2e-5, 0.3),    # Dropout 0.3
]

RESULTS_FILE = "/kaggle/working/results_s3.json"

# Load hasil yang sudah ada (untuk resume)
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r") as f:
        S3_RESULTS = json.load(f)
    print(f"Resume dari: {list(S3_RESULTS.keys())}")
else:
    S3_RESULTS = {}
    print("Mulai dari awal.")

# Tracker best model per task
BEST_MODEL_TRACKER = {}

In [ ]:
# Prepare splits (sama untuk semua konfigurasi)
S3_SPLITS = {}
for task_name, (text_col, label_col, label_names) in TASKS.items():
    texts  = df[text_col].values
    labels = df[label_col].values
    X_train, X_val, X_test, y_train, y_val, y_test = make_splits(texts, labels)
    S3_SPLITS[task_name] = (X_train, X_val, X_test, y_train, y_val, y_test)
    print(f"[{task_name}] Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")


# Training loop
for task_name, (text_col, label_col, label_names) in TASKS.items():
    print(f"\n=================== TRAINING SKENARIO 3 FOR TASK: {task_name} ===================")
    
    num_labels = len(label_names)
    X_train, X_val, X_test, y_train, y_val, y_test = S3_SPLITS[task_name]

    for hp_label, epochs, batch, lr, dropout in HYPERPARAM_GRID:
        for model_alias, model_name in S3_MODELS.items():
            result_key = f"{task_name}|{hp_label}|{model_alias}"

            # Skip jika sudah ada — tetap update tracker
            if task_name in S3_RESULTS:
                if hp_label in S3_RESULTS[task_name]:
                    if model_alias in S3_RESULTS[task_name][hp_label]:
                        print(f"  [SKIP] {result_key} sudah ada.")
                        existing_f1 = S3_RESULTS[task_name][hp_label][model_alias]["metrics"]["f1_macro"]
                        if task_name not in BEST_MODEL_TRACKER or existing_f1 > BEST_MODEL_TRACKER[task_name]["f1_macro"]:
                            BEST_MODEL_TRACKER[task_name] = {"f1_macro": existing_f1, "model_alias": model_alias,
                                                             "hp_label": hp_label, "model_name": model_name}
                        continue

            print(f"\n>>> {result_key} (ep={epochs}, bs={batch}, lr={lr}, dropout={dropout})")

            try:
                # Retry loading tokenizer dengan network resilience
                import time
                for attempt in range(3):
                    try:
                        tokenizer = AutoTokenizer.from_pretrained(model_name)
                        break
                    except Exception as e:
                        if attempt == 2:
                            raise e
                        print(f"  ⚠️ Gagal download tokenizer {model_name}, mencoba lagi ({attempt+2}/3) dalam 5 detik... (Error: {e})")
                        time.sleep(5)

                train_ds = build_hf_dataset(X_train, y_train, tokenizer)
                val_ds   = build_hf_dataset(X_val,   y_val,   tokenizer)

                output_dir = f"/kaggle/working/s3_{task_name}_{hp_label}_{model_alias}"
                hyperparams = {"epochs": epochs, "batch": batch, "lr": lr, "dropout": dropout}

                trainer, metrics, report, preds = train_and_evaluate(
                    model_name, train_ds, val_ds, X_test, y_test,
                    tokenizer, num_labels, label_names,
                    output_dir=output_dir,
                    num_epochs=epochs,
                    batch_size=batch,
                    lr=lr,
                    dropout=dropout,
                )

                print(f"  F1 Macro: {metrics['f1_macro']} | Accuracy: {metrics['accuracy']}")

                # Simpan hasil
                if task_name not in S3_RESULTS:
                    S3_RESULTS[task_name] = {}
                if hp_label not in S3_RESULTS[task_name]:
                    S3_RESULTS[task_name][hp_label] = {}

                S3_RESULTS[task_name][hp_label][model_alias] = {
                    "metrics": metrics,
                    "report":  report,
                    "preds":   preds.tolist(),
                    "hyperparams": hyperparams,
                }

                # ── Cek & simpan best model ──────────────────────────────────
                current_f1 = metrics["f1_macro"]
                if task_name not in BEST_MODEL_TRACKER or current_f1 > BEST_MODEL_TRACKER[task_name]["f1_macro"]:
                    BEST_MODEL_TRACKER[task_name] = {
                        "f1_macro": current_f1,
                        "model_alias": model_alias,
                        "hp_label": hp_label,
                        "model_name": model_name,
                    }
                    save_best_model(trainer, tokenizer, task_name, model_alias, model_name,
                                    hp_label, hyperparams, metrics, label_names, skenario="3")
                else:
                    print(f"  (bukan best: {current_f1:.4f} < {BEST_MODEL_TRACKER[task_name]['f1_macro']:.4f})")

                with open(RESULTS_FILE, "w") as f:
                    json.dump(S3_RESULTS, f, indent=2)
                print(f"  [SAVED] → {RESULTS_FILE}")

                # Bebaskan memori
                del trainer, tokenizer, train_ds, val_ds
                free_memory()

            except Exception as e:
                print(f"  [ERROR] {result_key}: {e}")
                free_memory()

print("\n✅ Hyperparameter tuning selesai!")
print("\n🏆 Best Models per Task:")
for task, info in BEST_MODEL_TRACKER.items():
    print(f"  {task}: {info['model_alias']} @ {info['hp_label']} (f1_macro={info['f1_macro']:.4f}) → /kaggle/working/best_model_s3_{task}/")

In [ ]:
# Tabel perbandingan
rows = []
for task_name, hp_configs in S3_RESULTS.items():
    if task_name == "ensemble":
        continue
    for hp_label, models in hp_configs.items():
        for model_alias, result in models.items():
            if "metrics" not in result:
                continue
            row = {"Task": task_name, "HP": hp_label, "Model": model_alias}
            row.update(result["metrics"])
            if "hyperparams" in result:
                row.update(result["hyperparams"])
            rows.append(row)

df_results_s3 = pd.DataFrame(rows)
print("\nTabel Perbandingan — Skenario 3 (Hyperparameter Tuning)")
if not df_results_s3.empty:
    display(df_results_s3.sort_values(["Task", "f1_macro"], ascending=[True, False])
                           .reset_index(drop=True))
else:
    print("⚠️ [WARNING] df_results_s3 kosong. Belum ada model yang berhasil dilatih atau disimpan di S3_RESULTS.")


In [ ]:
# ── ENSEMBLE: Majority Voting ──────────────────────────────────────────
def majority_voting(list_of_preds):
    """Majority voting dari beberapa prediksi (list of arrays)."""
    preds_stack = np.stack(list_of_preds, axis=1)
    ensemble_preds = []
    for row in preds_stack:
        counts = Counter(row)
        ensemble_preds.append(counts.most_common(1)[0][0])
    return np.array(ensemble_preds)


print("\n=================== ENSEMBLE — Skenario 3 ===================")

for task_name, (text_col, label_col, label_names) in TASKS.items():
    print(f"\n[{task_name}] Ensemble dari semua model × HP konfigurasi")
    
    all_preds = []
    model_labels = []
    
    if task_name not in S3_RESULTS:
        print(f"  Belum ada hasil untuk {task_name}, skip.")
        continue

    for hp_label, models in S3_RESULTS[task_name].items():
        if not isinstance(models, dict):
            continue
        for model_alias, result in models.items():
            if "preds" in result:
                all_preds.append(np.array(result["preds"]))
                model_labels.append(f"{hp_label}|{model_alias}")

    if len(all_preds) < 2:
        print(f"  Tidak cukup model untuk ensemble (butuh minimal 2).")
        continue

    X_train, X_val, X_test, y_train, y_val, y_test = S3_SPLITS[task_name]

    ensemble_preds = majority_voting(all_preds)
    ens_acc = accuracy_score(y_test, ensemble_preds)
    ens_f1m = f1_score(y_test, ensemble_preds, average="macro")
    ens_f1w = f1_score(y_test, ensemble_preds, average="weighted")

    print(f"\n{'='*55}")
    print(f"  ENSEMBLE — {task_name} ({len(all_preds)} models)")
    print(f"{'='*55}")
    print(f"  accuracy   : {ens_acc:.4f}")
    print(f"  f1_macro   : {ens_f1m:.4f}")
    print(f"  f1_weighted: {ens_f1w:.4f}")
    print()
    print(classification_report(y_test, ensemble_preds, target_names=label_names))

    if "ensemble" not in S3_RESULTS:
        S3_RESULTS["ensemble"] = {}
    S3_RESULTS["ensemble"][task_name] = {
        "metrics": {
            "accuracy":    round(ens_acc, 4),
            "f1_macro":    round(ens_f1m, 4),
            "f1_weighted": round(ens_f1w, 4),
        },
        "n_models": len(all_preds),
        "model_list": model_labels,
    }

with open(RESULTS_FILE, "w") as f:
    json.dump(S3_RESULTS, f, indent=2)
print(f"\n[SAVED] Hasil ensemble → {RESULTS_FILE}")
print("\n✅ Skenario 3 selesai!")

In [ ]:
# Visualisasi: Best HP per model
if not df_results_s3.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, task in zip(axes, ["Emotion", "Sentiment"]):
        sub = df_results_s3[df_results_s3["Task"] == task]
        if not sub.empty:
            best_hp = sub.loc[sub.groupby("Model")["f1_macro"].idxmax()]
            ax.barh(best_hp["Model"] + " (" + best_hp["HP"] + ")", best_hp["f1_macro"])
            ax.set_title(f"Best HP Config F1 Macro — {task}")
            ax.set_xlabel("F1 Macro")
            ax.set_xlim(0, 1)
    plt.tight_layout()
    plt.savefig("/kaggle/working/s3_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Plot disimpan ke /kaggle/working/s3_comparison.png")

In [ ]:
# ── Contoh cara load best model yang sudah disimpan ──────────────────────
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# import json
#
# task = "Emotion"  # atau "Sentiment"
# best_dir = f"/kaggle/working/best_model_s3_{task}"
#
# with open(f"{best_dir}/best_model_info.json") as f:
#     info = json.load(f)
# print("Model terbaik:", info)
#
# tokenizer = AutoTokenizer.from_pretrained(best_dir)
# model = AutoModelForSequenceClassification.from_pretrained(best_dir)
print("Uncomment kode di atas untuk load best model.")